## BronzeWork Incremental
Incremental Bronze ingestion with rerun-safe watermark logic

###Step-1 - Imports and setup
This cell import the Pyspark and delta helpers used in the notebook,awitches to the correct catalog and make sure the Bronze schema exists before we start loading data

In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog databricks_ecom_project")

In [0]:
spark.sql("create schema if not exists bronze_schema")

## Step-2 - Bronze control Table
this table stores watermark for each source table

it helps the pipeline remember:
- the latest timestamp already processed
- the latest primarykey processed at the timestamp
- how many rows written in the latest run

This makes Bronze load incremental and rerun-safe

In [0]:
spark.sql("""
          create table if not exists databricks_ecom_project.bronze_schema.ingestion_control(
              layer string,
              table_name string,
              ts_col string,
              pk_col string,
              last_successful_ts timestamp,
              last_successful_pk bigint,
              last_run_id string,
              rows_written bigint,
              run_status string,
              updated_at timestamp

          )
          using delta
          
          """)

## Step-3 -- Source table configuration 
This cell defines which sourcetable will be loaded into bronze and which columns should be used as:
- **Primary key**
- **timestamp/watermark column**

it also creates a unique `bronze_run_id` for the current pipeline run

In [0]:
table_config = {
    "orders": { "pk_col" : "order_id", "ts_col": "updated_at"},
    "products" : {"pk_col": "product_id","ts_col":"updated_at"},
    "payments" : {"pk_col" : "payment_id", "ts_col":"processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("current Bronze run ID:", bronze_run_id)

## STEP4 - Helper Functions

This cell contains reusable functions:
- `get_last_successful_watermark()` reads the last processed watermark from the control table
- `upsert_bronze_control()` updates the control table after asuccessful Bronze load

In [0]:
def get_last_successful_watermark(table_name:str):
    ctrl = (
        spark.table("databricks_ecom_project.bronze_schema.ingestion_control")
        .filter(
            (f.col("layer") == "bronze")&
            (f.col("table_name") == "table_name")&
            (f.col("run_status") == "success")
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)

    )
    rows = ctrl.collect()
    if not rows:
        return None, None

    return rows[0]["last_successful_ts"],rows[0]["last_successful_pk"]


In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
    control_df =spark.createDataFrame(
        [
            (
                "bronze",
                table_name,
                ts_col,
                pk_col,
                last_ts,
                int(last_pk) if last_pk is not None  else None,
                run_id,
                int(rows_written),
                "success",
                datetime.now()
            
            )
        ],
        schema = """
        layer string,
        table_name string,
         ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )
    dt = DeltaTable.forName(spark,"databricks_ecom_project.bronze_schema.ingestion_control")
    (dt.alias("t")
        .merge(control_df.alias("s"),"t.table_name = s.table_name and t.layer = s.layer")
        .whenMatchedUpdate(set={
            "ts_col" : "s.ts_col",
            "pk_col" : "s.pk_col",
            "last_successful_ts":"s.last_successful_ts",
            "last_successful_pk":"s.last_successful_pk",
            "last_run_id":"s.last_run_id",
            "rows_written":"s.rows_written",
            "run_status":"s.run_status",
            "updated_at":"s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


### Step -5 Bronze incremental load loop
This is the main Bronze logic.
- reads the last watermark
- reads the source sql table
- filters only new/changed rows
- adds Bronze audit columns
- appends the rows INTO bronze Delta table
- updates the control table

This is the core incremental loading logic


In [0]:
from pyspark.sql import functions as F

for table_name, cfg in table_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]

    source_table = f"novacart_casestudy_catalog.dbo.{table_name}"
    target_table = f"databricks_ecom_project.bronze_schema.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)

    # Optional: truncate Python timestamp to milliseconds
    if last_successful_ts is not None:
        last_successful_ts = last_successful_ts.replace(
            microsecond=(last_successful_ts.microsecond // 1000) * 1000
        )

    print(f"\n=== Processing {table_name} ===")
    print(f"last successful ts: {last_successful_ts}")
    print(f"last successful pk: {last_successful_pk}")

    source_df = (
        spark.read
        .table(source_table)
        .withColumn(
            ts_col,
            F.date_trunc("MILLISECOND", F.col(ts_col).cast("timestamp"))
        )
    )

    if last_successful_ts is None:
        rows_to_load = source_df

    elif last_successful_pk is None:
        rows_to_load = source_df.filter(
            F.col(ts_col) > F.lit(last_successful_ts)
        )

    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) &
                (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
            )
        )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    rows_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {rows_count}")

    if rows_count == 0:
        print(f"No new rows for {table_name}.")

        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            rows_count,
            bronze_run_id
        )

        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    max_ts = (
        rows_to_load
        .agg(F.max(ts_col).alias("max_ts"))
        .collect()[0]["max_ts"]
    )

    max_pk = (
        rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        rows_count,
        bronze_run_id
    )

    print(f"Wrote {rows_count} rows to {target_table}")

In [0]:
print("orders Bronze count:",spark.sql("select count(*) from databricks_ecom_project.bronze_schema.orders_raw").collect()[0][0])
print("products Bronze count:",spark.sql("select count(*) from databricks_ecom_project.bronze_schema.products_raw").collect()[0][0])
print("payments Bronze count:",spark.sql("select count(*) from databricks_ecom_project.bronze_schema.payments_raw").collect()[0][0])

display(spark.sql("select * from databricks_ecom_project.bronze_schema.ingestion_control").orderBy("table_name"))

